# 🔍 Interactive YOLO Inference & Post-processing

This notebook provides an interactive UI to upload raw images, run YOLOv12 instance segmentation, apply morphological post-processing, and visualize the hollow contours mimicking the CVAT annotation style.

In [1]:
# Install required libraries if not present
!pip install -q ultralytics ipywidgets shapely opencv-python matplotlib

In [ ]:
import os
import sys
import logging
from pathlib import Path
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
from matplotlib import pyplot as plt

# Add root directory to python path to import our custom pipeline
sys.path.append(str(Path(os.path.abspath('')).parent.parent))

from src.inference import MaskPostProcessor, RockVisualizer
from ultralytics import YOLO

# Suppress Ultralytics logs globally to avoid UI console flooding (e.g. corrupt JPEG restored)
logging.getLogger("ultralytics").setLevel(logging.ERROR)

print("✅ Dependencies loaded successfully!")

✅ Dependencies loaded successfully!


## ⚙️ 1. Configurations & Model Loading
Ensure you specify the correct exact path to your trained model `best.pt`.

In [8]:
# Update this path to your locally trained model weights
weights_path = "/home/praktikan/projects/github/DwiAnggara/Automatic-Cutting-Description/notebooks/training/models/YOLO26m_Batch4_March_Dataset_20260406_132837/weights/best.pt"
# weights_path = "/home/praktikan/projects/github/DwiAnggara/Automatic-Cutting-Description/notebooks/training/models/YOLO26m_Batch4_March_Dataset_SAHI_20260407_121154/weights/best.pt"
# weights_path = "/home/praktikan/projects/github/DwiAnggara/Automatic-Cutting-Description/notebooks/training/models/Best_MasRG/best.pt"

try:
    print(f"Loading YOLO model from: {weights_path}")
    model = YOLO(weights_path)
    print("✅ Model loaded successfully!")
except Exception as e:
    print(f"❌ Error loading model! Check your path. Details: {e}")

Loading YOLO model from: /home/praktikan/projects/github/DwiAnggara/Automatic-Cutting-Description/notebooks/training/models/YOLO26m_Batch4_March_Dataset_20260406_132837/weights/best.pt
✅ Model loaded successfully!


## 🎛️ 2. Interactive Inference Setup
Run the cell below to display an interactive file uploader and slider controls for fine-tuning the segmentation algorithm.

In [ ]:
# Style and Layout
style = {'description_width': 'initial'}
layout = widgets.Layout(width='400px')

# UI Components
upload_btn = widgets.FileUpload(
    accept='image/*',
    multiple=False,
    description='Upload Image'
)

use_sahi_checkbox = widgets.Checkbox(
    value=False,
    description='Use SAHI (High Res / Slicing)',
    style=style, layout=layout
)

conf_slider = widgets.FloatSlider(
    value=0.25, min=0.05, max=0.95, step=0.05,
    description='Confidence:', style=style, layout=layout
)

iou_slider = widgets.FloatSlider(
    value=0.45, min=0.1, max=0.9, step=0.05,
    description='NMS IoU:', style=style, layout=layout
)

min_area_slider = widgets.IntSlider(
    value=300, min=0, max=5000, step=50,
    description='Min Area (px²):', style=style, layout=layout
)

epsilon_slider = widgets.FloatSlider(
    value=0.01, min=0.001, max=0.05, step=0.001, readout_format='.3f',
    description='Epsilon Ratio:', style=style, layout=layout
)

run_button = widgets.Button(
    description='Run Inference',
    button_style='success',
    icon='play'
)

output_plot = widgets.Output()

def run_interactive_inference(b):
    with output_plot:
        clear_output(wait=True)
        
        if not upload_btn.value:
            print("⚠️ Please upload an image first.")
            return
            
        print("⏳ Processing...")
        
        # 1. Read Uploaded Image
        uploaded_file = list(upload_btn.value.values())[0] if isinstance(upload_btn.value, dict) else upload_btn.value[0]
        content = uploaded_file['content']
        
        # Decode byte array to cv2 image
        nparr = np.frombuffer(content, np.uint8)
        original_bgr = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
        h, w = original_bgr.shape[:2]
        
        raw_polygons_by_class = {}
        proc_polygons_by_class = {}
        
        post_processor = MaskPostProcessor(
            min_area=min_area_slider.value,
            epsilon_ratio=epsilon_slider.value,
            simplify_tol=2.0
        )
        
        # 2. Run YOLO Inference (SAHI or Normal)
        if use_sahi_checkbox.value:
            from sahi import AutoDetectionModel
            from sahi.predict import get_sliced_prediction
            import torch
            
            device_mode = "cuda:0" if torch.cuda.is_available() else "cpu"
            detection_model = AutoDetectionModel.from_pretrained(
                model_type='yolov8',
                model_path=model.ckpt_path if hasattr(model, 'ckpt_path') else model.pt_path, 
                confidence_threshold=conf_slider.value,
                device=device_mode
            )
            result_sahi = get_sliced_prediction(
                original_bgr,
                detection_model,
                slice_height=512,
                slice_width=512,
                overlap_height_ratio=0.2,
                overlap_width_ratio=0.2,
                verbose=False
            )
            
            for obj in result_sahi.object_prediction_list:
                if obj.mask is not None:
                    class_id_int = obj.category.id
                    
                    # Convert boolean mask from SAHI natively to 255-based uint8 mask
                    mask_array = (obj.mask.bool_mask * 255).astype(np.uint8)
                    
                    # Compute raw polygon directly from the boolean mask for "Raw Visualization"
                    contours, _ = cv2.findContours(mask_array, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                    if class_id_int not in raw_polygons_by_class:
                        raw_polygons_by_class[class_id_int] = []
                        
                    for cnt in contours:
                        if len(cnt) > 2:
                            raw_polygons_by_class[class_id_int].append(cnt.reshape(-1, 2))
                    
                    # Process clean polygons logically matching inference bounds
                    clean_polys = post_processor.process(mask_array)
                    
                    if class_id_int not in proc_polygons_by_class:
                        proc_polygons_by_class[class_id_int] = []
                    proc_polygons_by_class[class_id_int].extend(clean_polys)
                    
        else:
            # Native YOLO Inference
            results = model.predict(
                source=original_bgr, 
                conf=conf_slider.value, 
                iou=iou_slider.value, 
                verbose=False
            )[0]
            
            if results.masks is not None:
                mask_xys = results.masks.xy
                class_ids = results.boxes.cls.cpu().numpy()
                
                for mask_xy, cls_id in zip(mask_xys, class_ids):
                    if len(mask_xy) > 2:
                        class_id_int = int(cls_id)
                        if class_id_int not in raw_polygons_by_class:
                            raw_polygons_by_class[class_id_int] = []
                        
                        mask_xy_int = mask_xy.astype(np.int32)
                        raw_polygons_by_class[class_id_int].append(mask_xy_int)

                        canvas = np.zeros((h, w), dtype=np.uint8)
                        cv2.fillPoly(canvas, [mask_xy_int], 255)
                        
                        clean_polys = post_processor.process(canvas)
                        
                        if class_id_int not in proc_polygons_by_class:
                            proc_polygons_by_class[class_id_int] = []
                        proc_polygons_by_class[class_id_int].extend(clean_polys)
                
        # 4. Render Visualizations 
        visualizer = RockVisualizer(thickness=8)
        
        raw_hollow = visualizer.draw_hollow(original_bgr, raw_polygons_by_class)
        raw_solid = visualizer.draw_mask_only(original_bgr.shape, raw_polygons_by_class)
        
        proc_hollow = visualizer.draw_hollow(original_bgr, proc_polygons_by_class)
        proc_solid = visualizer.draw_mask_only(original_bgr.shape, proc_polygons_by_class)
        
        # 5. Display 2x3 Grid
        if hasattr(visualizer, 'plot_advanced_comparison'):
            fig = visualizer.plot_advanced_comparison(original_bgr, raw_hollow, proc_hollow, raw_solid, proc_solid, title_suffix="")
        else:
            fig = visualizer.plot_comparison(original_bgr, proc_hollow, proc_solid, title_suffix="")
            
        display(fig)
        plt.close(fig)
        print("✅ Processing complete!")

# Attach click event
run_button.on_click(run_interactive_inference)

# Display UI Structure
ui = widgets.VBox([
    widgets.HTML("<b>Upload Sample Image</b>"),
    upload_btn,
    widgets.HTML("<hr><b>Model Settings</b>"),
    use_sahi_checkbox,
    conf_slider,
    iou_slider,
    widgets.HTML("<hr><b>Post-processing Settings (Smoothing & De-noising)</b>"),
    min_area_slider,
    epsilon_slider,
    widgets.HTML("<br>"),
    run_button,
    output_plot
])

display(ui)